# Real-time benchmark: results

This notebook reports the primary test and the secondary analysis of the preregistered real-time benchmark of LLM twins
against simple models. The plan, `docs/prereg/realtime-benchmark.md`, is frozen by the git tag
`prereg/realtime-benchmark` and registered at https://osf.io/gh42n. It defines the forecast task, the candidates, the
procedure and every statistic below. The statistics come from the run `experiments/realtime/2026-11-22-benchmark`,
which computed them once from the twin probabilities of `experiments/realtime/2026-11-22-twin-web`.

The run departs in one respect from the code that the tag froze. The data release it uses, downloaded on 2026-11-21,
holds the code 7 for `NUMADT` in 3 web interviews of August to October 2024, a code that the cleaning at the tag did not
accept. The cleaning at the run's commit reads it as 5 or more (commit `c2fe1ec`; `data/README.md`).

In [1]:
import json
import pandas as pd
from sdt.realtime import RUNS

RUN = RUNS / '2026-11-22-benchmark'
primary = json.load(open(RUN / 'primary.json'))
secondary = json.load(open(RUN / 'secondary.json'))
meta = json.load(open(RUN / 'run.json'))

# Claim checks for the text of sections 1 and 2
assert meta['twins']['name'] == '2026-11-22-twin-web' and meta['units_with_errors'] == {}
assert primary['T'] == 26 and primary['df'] == 4 and primary['outcome'] == 'neither shown more accurate'
assert [round(primary[k], 4) for k in ['mean', 'lo', 'hi']] == [0.0026, -0.0045, 0.0097]
assert [round(100 * primary['as_share_of_avoidable_loss'][k]) for k in ['mean', 'lo', 'hi']] == [28, -49, 104]
assert round(primary['statistic'], 2) == 1.01 and round(primary['p'], 2) == 0.37
assert len(secondary['months']) == 16 and (secondary['months'][0], secondary['months'][-1]) == (202506, 202609)
assert [round(secondary['rho'][k], 2) for k in ['estimate', 'lo', 'hi']] == [0.85, -0.23, 0.91]
assert [round(secondary['slope'][k], 5) for k in ['estimate', 'lo', 'hi']] == [0.00059, -0.00004, 0.0018]
assert round(secondary['breakeven']['estimate'], 1) == 5.0 and secondary['breakeven']['lo'] == 'below 1'
assert round(secondary['breakeven']['hi'], 1) == 9.1

## 1. Primary test

Over the 26 target months from August 2024 to September 2026, at a horizon of two months, d̄ is 0.0026, with a 95%
interval from −0.0045 to 0.0097. As a share of the simple pick's avoidable loss, d̄ is 28%, with an interval from −49%
to 104%. A positive d̄ means that the twin pick's loss is the lower. The statistic is 1.01 with 4 degrees of freedom,
so p = 0.37. By the decision rule of section 7 of the plan, neither class is shown more accurate.

The table also reports the two subperiods of section 7, to which the plan attaches no decision rule.

In [2]:
periods = {'August 2024 to September 2026': primary,
           'August 2024 to February 2026 (known)': primary['subperiods']['known'],
           'March to September 2026 (unseen)': primary['subperiods']['unseen']}
share = lambda r, k: f"{100 * r['as_share_of_avoidable_loss'][k]:.0f}%"
pd.DataFrame({name: {'months': r['T'], 'd̄': f"{r['mean']:.4f}", '95% interval': f"{r['lo']:.4f} to {r['hi']:.4f}",
                     'd̄, share of avoidable loss': share(r, 'mean'),
                     'interval, share of avoidable loss': f"{share(r, 'lo')} to {share(r, 'hi')}",
                     'statistic': f"{r['statistic']:.2f}", 'degrees of freedom': r['df'], 'p': f"{r['p']:.2f}"}
              for name, r in periods.items()})

,August 2024 to September 2026,August 2024 to February 2026 (known),March to September 2026 (unseen)
months,26,19,7
d̄,0.0026,0.0040,-0.0011
95% interval,-0.0045 to 0.0097,-0.0056 to 0.0136,-0.0069 to 0.0047
"d̄, share of avoidable loss",28%,33%,-44%
"interval, share of avoidable loss",-49% to 104%,-48% to 115%,-270% to 182%
statistic,1.01,1.14,-0.84
degrees of freedom,4,4,2
p,0.37,0.32,0.49


## 2. Secondary analysis

Over the 16 target months from June 2025 to September 2026, d̄(h) is computed for horizons h of 1 to 12 months. Its
Spearman correlation with the horizon, ρ, is 0.85, with a 95% interval from −0.23 to 0.91. Its least-squares slope is
0.00059 per month of horizon, with an interval from −0.00004 to 0.00180. Its breakeven horizon is 5.0 months, with an
interval from below 1 to 9.1. Section 8 of the plan attaches no decision rule to these quantities.

In [3]:
curve = pd.Series(secondary['curve']).rename(lambda h: int(h)).rename_axis('horizon h (months)')
display(curve.map(lambda x: f'{x:.4f}').rename('d̄(h)').to_frame().T)
show = lambda v, f: v if isinstance(v, str) else format(v, f)
pd.DataFrame({name: {'estimate': show(secondary[k]['estimate'], f), '95% interval': f"{show(secondary[k]['lo'], f)} to "
                                                                                    f"{show(secondary[k]['hi'], f)}"}
              for name, k, f in [('ρ', 'rho', '.2f'), ('slope per month of horizon', 'slope', '.5f'),
                                 ('breakeven horizon (months)', 'breakeven', '.1f')]}).T

horizon h (months),1,2,3,4,5,6,7,8,9,10,11,12
d̄(h),-0.0001,-0.0002,-0.0004,-0.0012,-0.0001,0.0031,0.0034,0.0044,0.0045,0.0045,0.0048,0.0041


,estimate,95% interval
ρ,0.85,-0.23 to 0.91
slope per month of horizon,0.00059,-0.00004 to 0.00180
breakeven horizon (months),5.0,below 1 to 9.1
